# Classification performance scripts:
This notebook consists of two scripts: 
- "Classification performance measuring" for measuring the performance of LLM classification against the ground truth using metrics: Macro F1 & Hamming Loss
- "Classification model comparison" for creating a .csv file and figure comparing performance of all used model configurations

**Classification performance measuring script:**

In [1]:
import pandas as pd
import sys
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, hamming_loss
import warnings

# Set font and do not output graphs in Jupyter notebook
matplotlib.use('Agg')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.labelsize'] = 12

configuration:

In [ ]:
# File names and directories
ROOT = "--FILL IN YOUR ROOT FOLDER--"  # root project folder
CLSF_DIR = f"{ROOT}/LLM Analyses/Classification/Ground Truth sample - clsf/Classifications"
GROUND_TRUTH_DIR = f"{ROOT}/Ground Truth/Annotations"
OUTPUT_DIR = f"{ROOT}/LLM Analyses/Classification/Ground Truth sample - clsf/Performance measuring"

GROUND_TRUTH_FILE = "Ground truth - clsf.csv"  # Manually classified csv
OUTPUT_METRICS = "Clsf performance"
OUTPUT_CONF_MATRICES = "Clsf conf matrices"

# LSEG categories
CATEGORIES = ["Cat_E", "Cat_S", "Cat_G", "Cat_N"]
SUBCATEGORIES = [
    "Resource_Use", "Emissions", "Innovation", 
    "Workforce", "Human_Rights", "Community", "Product_Responsibility", 
    "Management", "Shareholders", "CSR_Strategy"
]

main algorithm:

In [ ]:
def load_and_prepare_data(clsf_file_name):
    """Loads the files and aligns them using the 'Link' column."""
    clsf_file = os.path.join(CLSF_DIR, clsf_file_name)
    gt_file = os.path.join(GROUND_TRUTH_DIR, GROUND_TRUTH_FILE)

    if not os.path.exists(clsf_file): sys.exit(f"LLM input file is missing: {clsf_file_name}")        
    if not os.path.exists(gt_file): sys.exit(f"Ground Truth input file is missing: {GROUND_TRUTH_FILE}")  

    df_pred = pd.read_csv(clsf_file)
    df_true = pd.read_csv(gt_file)

    # Clean column names to avoid whitespace issues
    df_pred.columns = df_pred.columns.str.strip()
    df_true.columns = df_true.columns.str.strip()

    # Align data by sorting by Link
    df_pred = df_pred.sort_values(by="URL").reset_index(drop=True)
    df_true = df_true.sort_values(by="URL").reset_index(drop=True)

    if len(df_pred) != len(df_true):
        print(f"WARNING: Row counts do not match ({len(df_pred)} vs {len(df_true)}). Performing inner join...")
        combined = pd.merge(df_true, df_pred, on="URL", suffixes=('_true', '_pred'))
        return combined
    
    return pd.concat([df_true.add_suffix('_true'), df_pred.add_suffix('_pred')], axis=1)

def df_check(clsf_file_name):
    """Checks if the LLM dataframe contains any error messages."""
    df = pd.read_csv(os.path.join(CLSF_DIR, clsf_file_name))
    for idx, value in enumerate(df["Cat_E"]):
        val_clean = str(value).strip().upper()
        if "ERROR" in val_clean:
            print(f"{clsf_file_name} [Row {idx}]: contains ERROR message")

def get_binary_values(series):
    """Converts 'Yes'/'No' to binary integers."""
    return series.astype(str).str.strip().str.capitalize().map({'Yes': 1, 'No': 0}).astype(int)

def find_column(df_cols, target_base, suffix):
    """Finds a column name in a list regardless of case sensitivity."""
    normalized_target = f"{target_base}_{suffix}".lower().replace(" ", "_")
    for col in df_cols:
        if col.lower().replace(" ", "_") == normalized_target:
            return col
    return None

def measure_performance(clsf_file, output_metrics, output_conf_matrices):
    # Ignore specific RuntimeWarnings in single-class cases
    warnings.filterwarnings("ignore", category=RuntimeWarning, message="invalid value encountered in scalar divide")
    
    data = load_and_prepare_data(clsf_file)
    
    results = []    
    
    # We will split metrics collection into two groups
    sub_true_labels = []
    sub_pred_labels = []

    # All targets for individual reporting
    target_list = [(cat, cat, "Main") for cat in CATEGORIES] + [(sub, f"{sub}_active", "Sub") for sub in SUBCATEGORIES]

    # Prepare visualization
    n_cols = 3
    n_rows = (len(target_list) + 1 + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten()

    for i, (display_name, base_name, cat_type) in enumerate(target_list):
        # Look for the true and predicted versions in the combined dataframe
        true_col = find_column(data.columns, base_name, "true")
        pred_col = find_column(data.columns, base_name, "pred")

        if not true_col or not pred_col:
            print(f"Skipping {display_name}: Not found.")
            continue

        y_true = get_binary_values(data[true_col])
        y_pred = get_binary_values(data[pred_col])

        # ONLY add to global calculation if it's a subcategory
        if cat_type == "Sub":
            sub_true_labels.append(y_true)
            sub_pred_labels.append(y_pred)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        results.append({
            "Type": cat_type,
            "Category": display_name,
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "Ground_Truth_Yes": int(sum(y_true)),
            "LLM_Classifier_Yes": int(sum(y_pred)),
            "Accuracy": acc,
            "F1_Score": f1, # F1 score per (sub)category
        })

        sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', ax=axes[i], cmap='Greens', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
        axes[i].set_title(display_name.replace("_", " ")) # Dutch titles

    # Global Matrix, Macro F1 and Hamming Loss are based ONLY on Subcategories
    if sub_true_labels:
        # Convert the matrices to shape: [number of documents (N), number of subcategories (10)]
        Y_true_sub = np.stack(sub_true_labels, axis=1)
        Y_pred_sub = np.stack(sub_pred_labels, axis=1)

        # 1. Flattened version for global confusion matrix
        y_t_sub_flat = Y_true_sub.flatten()
        y_p_sub_flat = Y_pred_sub.flatten()
        g_tn, g_fp, g_fn, g_tp = confusion_matrix(y_t_sub_flat, y_p_sub_flat, labels=[0, 1]).ravel()
        
        sns.heatmap([[g_tn, g_fp], [g_fn, g_tp]], annot=True, fmt='d', ax=axes[i+1], cmap='Blues', xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'])
        axes[i+1].set_title("Globale confusion matrix") # Dutch titles

        # Hamming Loss
        h_loss = hamming_loss(Y_true_sub, Y_pred_sub)

        # Individual F1 scores per subcategory
        indiv_f1_scores = f1_score(Y_true_sub, Y_pred_sub, average=None, zero_division=0)

        # Macro F1 is calculated by averaging the F1 score per subcategory
        macro_f1 = np.mean(indiv_f1_scores)

        # Results dataframe with summary metrics
        df_results = pd.DataFrame(results)
        summary_row = pd.DataFrame([{
            "Category": "SUBCATEGORY_GLOBAL_METRICS",           
            "TP": int(g_tp), "FP": int(g_fp), "TN": int(g_tn), "FN": int(g_fn),
            "Note": f"Hamming Loss: {h_loss:.4f} | Macro F1: {macro_f1:.4f} | Accuracy: {accuracy_score(y_t_sub_flat, y_p_sub_flat):.4f}"
        }])
        
        pd.concat([df_results, summary_row], ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, output_metrics), index=False, encoding='utf-8-sig')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, output_conf_matrices))
        print(f"Success.")

def main():
    """Iterates over all classification outputs in given directory"""
    for clsf_file in os.listdir(CLSF_DIR):
        if "Classification" in clsf_file:
            suffix = clsf_file.replace("Classification", "").replace(".csv", "")
            if OUTPUT_METRICS + suffix + ".csv" not in os.listdir(OUTPUT_DIR):
                df_check(clsf_file)
                print(f"Measuring performance for {suffix.replace("_", " ")} Model")
                measure_performance(clsf_file, OUTPUT_METRICS + suffix + ".csv", OUTPUT_CONF_MATRICES + suffix + ".png")
            else:
                print(f"Performance for {suffix.replace("_", " ")} Model has already been measured")
    print("Finished.")

if __name__ == "__main__":
    main()

**Model comparison script:**

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd
import os

configuration:

In [5]:
INPUT_DIR = "./Performance measuring"
OUTPUT_DIR = "."
FILE_PATTERN = r"Clsf performance.*\.csv"
OUTPUT_FILE = "Clsf model performance comparison"   # file name without extension

main algorithm:

In [ ]:
def extract_global_metrics(filepath):
    """
    Extracts global metrics from the performance measure .csv.
    """
    try:
        df = pd.read_csv(filepath)
        filename = os.path.basename(filepath)
        
        summary = df[df['Category'] == 'SUBCATEGORY_GLOBAL_METRICS']
        if summary.empty:
            print(f"Warning: No 'SUBCATEGORY_GLOBAL_METRICS' row found in {filename}. Skipping.")
            return None
        
        # Extract Hamming Loss, Macro F1 & Accuracy from the 'Note' column using regex
        note = str(summary['Note'].values[0])

        h_loss_match = re.search(r"Hamming Loss: ([\d\.]+)", note)
        h_loss = float(h_loss_match.group(1)) if h_loss_match else None
        
        f1_macro_match = re.search(r"Macro F1: ([\d\.]+)", note)
        f1_macro = float(f1_macro_match.group(1)) if f1_macro_match else None

        acc_match = re.search(r"Accuracy: ([\d\.]+)", note)
        acc = float(acc_match.group(1)) if acc_match else None

        # Clean up filename for the legend
        display_name = filename.replace(".csv", "").replace("Clsf performance", "").lstrip("_").replace("_", " ")
        # Alternative naming to be used in thesis scripture:
        display_name = display_name.replace("role", "rol ").replace("examples", "few-shot").replace("reasoning", "CoT").replace(" rol 0", "")
        display_name = display_name.replace("selfc", "zelfreflectie").replace("GPT5n", "GPT-5-nano").replace("GPT5.1", "GPT-5.1")
        if display_name in ("GPT-5.1", "GPT-5-nano"): display_name += " baseline"

        return {
            "Model": display_name,
            "Macro F1": f1_macro,
            "Hamming Loss": h_loss,
            "Accuracy": acc
        }
    
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None

def main():
   
    # 1. Find all relevant files in the subfolder
    all_files = [f for f in os.listdir(INPUT_DIR) if re.match(FILE_PATTERN, f)]

    if not all_files:
        print("No validation files found matching the pattern and filters.")
        return

    print(f"Found {len(all_files)} potential files. Extracting subcategory metrics...")

    # 3. Extract metrics
    comparison_data = []
    for file in all_files:
        filepath = os.path.join(INPUT_DIR, file)
        metrics = extract_global_metrics(filepath)
        if metrics:
            comparison_data.append(metrics)

    if not comparison_data:
        print("No compatible data found.")
        return

    df_comp = pd.DataFrame(comparison_data)

    # 4. Determine the best model
    best_model_idx = df_comp['Macro F1'].idxmax()   # General best model is the one with highest Macro F1
    best_model = df_comp.loc[best_model_idx]
    best_hl_idx = df_comp['Hamming Loss'].idxmin()    # Model with best Hamming Loss
    best_hl = df_comp.loc[best_hl_idx]

    print("\n--- PERFORMANCE RANKING ---")
    print(df_comp.sort_values(by="Macro F1", ascending=False).to_string(index=False))
    
    print(f"\nBEST MODEL CONFIGURATION: {best_model['Model']}")
    print(f"Best Hamming Loss : {best_hl['Hamming Loss']:.4f}")

    # Save csv
    df_comp.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_FILE + ".csv"))
    print(f"\nComparison summary saved as: {OUTPUT_FILE + ".csv"}")


    # 5. Visualization
    df_comp.drop(columns=['Accuracy'], inplace=True)
    df_melted = df_comp.melt(id_vars="Model", var_name="Metric", value_name="Score")

    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")
    
    # Create the plot
    color_palette = ["#1E64C8", "#FFD200"]  # University of Ghent color palette
    ax = sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", palette=color_palette)
    
    # Axis labels are Dutch to use in thesis scripture
    plt.title("Classificatie: modelvergelijking", fontsize=16, pad=20)
    plt.ylabel("Score", fontsize=12)
    plt.xlabel("Modelconfiguratie", fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    
    # Add values on top of bars
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:.3f}', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='center', 
                        xytext=(0, 10), 
                        textcoords='offset points',
                        fontsize=9,
                        fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, OUTPUT_FILE + ".svg"))
    print(f"\nComparison graph saved as: {OUTPUT_FILE + ".svg"}")

if __name__ == "__main__":
    main()